# PPPD Vestibular Rehabilitation Outcome Prediction — Exploratory Analysis

**Project:** Clinical Decision Support System for Persistent Postural-Perceptual Dizziness (PPPD)  
**Author:** Erin Chan  
**Date:** March 2026

> **Note:** Cell outputs have been cleared to keep the repository lightweight. To reproduce all figures and tables, run **Kernel → Restart & Run All** (or `Cell → Run All` in VS Code). Total runtime is approximately 30–60 seconds.

## Project Overview

This notebook contains the exploratory data analysis (EDA) and model evaluation for a machine learning system that predicts vestibular rehabilitation therapy (VRT) outcomes in PPPD patients. The model takes 8 clinical features and predicts **treatment response rates** (% improvement in DHI score) for both traditional VRT and VR-enhanced VRT.

### Data Source
- **Demographics:** OpenNeuro ds004460 v1.1.0 (Gramann et al., 2021) — 20 healthy participants
- **Clinical variables:** Simulated from published PPPD literature distributions, bootstrapped to n=500
- **Why simulated?** No public clinical PPPD outcome dataset with DHI scores, treatment outcomes, and patient-level covariates exists. The simulation is grounded in published effect sizes and cohort statistics, making it a reasonable proxy for demonstrating the ML pipeline.

### Key References
- Staab et al. (2017) — PPPD diagnostic criteria
- Steensnaes et al. (2023) — VRT outcomes in PPPD
- Micarelli et al. (2019) — VR-enhanced vestibular rehabilitation
- Popkirov et al. (2018) — Anxiety-dizziness relationship
- Bittar & von Söhsten Lins (2015) — Symptom duration and prognosis
- Herdman et al. (2020) — Comorbidity prevalence in PPPD

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
SEED = 42
np.random.seed(SEED)

print("Libraries loaded successfully.")

## 2. Load and Inspect the Dataset

The training dataset was generated by `train_model.py`, which:
1. Loads real demographics from OpenNeuro ds004460 (20 participants with age, sex, handedness)
2. Bootstrap up-samples to n=500 with age jitter
3. Simulates clinical variables (DHI, anxiety, visual sensitivity, symptom duration, trigger count, comorbidities) from published literature distributions
4. Simulates treatment response rates for traditional VRT and VR-enhanced VRT

This is an intentional design choice: no public dataset exists with PPPD patient-level DHI scores and rehabilitation outcomes. The simulation is grounded in published cohort statistics and effect sizes, making it suitable for demonstrating ML methodology while being transparent about its synthetic nature.

In [ ]:
df = pd.read_csv("training_data.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")
df.head(10)

In [ ]:
df.info()

## 3. Data Quality Assessment and Summary Statistics

Before building any model, we need to understand the data's quality and distributional properties. This includes checking for missing values (which were intentionally introduced at 5% for anxiety and visual sensitivity to simulate real clinical data), examining outliers, and reviewing summary statistics against the published literature values that informed the simulation.

In [ ]:
# Summary statistics
df.describe().round(3)

In [ ]:
# Missing values check
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
# Boxplots for outlier detection
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
feature_cols = ["age", "baseline_dhi", "anxiety", "visual_sens",
                "symptom_duration", "trigger_count", "migraine", "anxiety_disorder"]

for i, col in enumerate(feature_cols):
    ax = axes[i // 4, i % 4]
    sns.boxplot(y=df[col], ax=ax, color="#3b82f6", width=0.4)
    ax.set_title(col.replace("_", " ").title(), fontsize=11)

plt.suptitle("Boxplots — Outlier Detection for All Features", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- No missing values in the training CSV (imputation was handled during `train_model.py`)
- `baseline_dhi` ranges roughly 20–85, with a mean near 52 — consistent with Staab et al. (2017) reporting mean DHI of ~52 in PPPD cohorts
- `anxiety` and `visual_sens` are on a 0–10 scale, centered around 5
- `symptom_duration` shows a right-skewed log-normal distribution (range 1–72 months), reflecting that most patients present within the first 1–2 years but some have chronic symptoms
- `migraine` (~35% prevalence) and `anxiety_disorder` (~40%) match Herdman et al. (2020) comorbidity rates
- No extreme outliers requiring removal

## 4. Distribution Plots of Key Features

Understanding the shape of each feature helps identify potential non-linearities the model needs to capture and validates that the simulation produces clinically reasonable distributions.

In [ ]:
# Distribution plots for continuous features
continuous = ["age", "baseline_dhi", "anxiety", "visual_sens", "symptom_duration"]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(continuous):
    ax = axes[i]
    sns.histplot(df[col], kde=True, ax=ax, color="#3b82f6", edgecolor="white", bins=25)
    ax.axvline(df[col].mean(), color="#ef4444", linestyle="--", label=f"Mean: {df[col].mean():.1f}")
    ax.axvline(df[col].median(), color="#10b981", linestyle="--", label=f"Median: {df[col].median():.1f}")
    ax.set_title(col.replace("_", " ").title(), fontsize=12)
    ax.legend(fontsize=9)

# Remove empty subplot
axes[5].set_visible(False)
plt.suptitle("Distributions of Continuous Features", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of categorical / discrete features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.countplot(x="trigger_count", data=df, ax=axes[0], color="#3b82f6")
axes[0].set_title("Trigger Count Distribution")

for i, col in enumerate(["migraine", "anxiety_disorder"]):
    counts = df[col].value_counts().sort_index()
    axes[i + 1].bar(["No (0)", "Yes (1)"], counts.values, color=["#94a3b8", "#ef4444"])
    axes[i + 1].set_title(col.replace("_", " ").title())
    for j, v in enumerate(counts.values):
        axes[i + 1].text(j, v + 5, f"{v} ({v/len(df)*100:.0f}%)", ha="center", fontsize=10)

plt.suptitle("Distributions of Discrete & Binary Features", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Target variable distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(["vrt_response", "vr_response"]):
    sns.histplot(df[col], kde=True, ax=axes[i], color=["#3b82f6", "#10b981"][i], bins=30)
    axes[i].axvline(df[col].mean(), color="#ef4444", linestyle="--",
                     label=f"Mean: {df[col].mean():.3f}")
    axes[i].set_title(f"{'Traditional VRT' if 'vrt' in col else 'VR-Based VRT'} Response Rate")
    axes[i].set_xlabel("Response Rate (proportion of DHI improvement)")
    axes[i].legend()

plt.suptitle("Target Variable Distributions — Treatment Response Rates", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Mean VRT response rate:  {df['vrt_response'].mean():.3f} ({df['vrt_response'].mean()*100:.1f}% DHI reduction)")
print(f"Mean VR response rate:   {df['vr_response'].mean():.3f} ({df['vr_response'].mean()*100:.1f}% DHI reduction)")

**Key Distribution Insights:**

- **Symptom duration** has a right-skewed distribution (log-normal), consistent with clinical observations — most patients seek treatment within 6–18 months, but some present after years of chronic dizziness
- **Trigger count** centres around 2–3, matching the typical multi-trigger nature of PPPD
- **Migraine** prevalence is ~35% and **anxiety disorder** ~40%, closely matching Herdman et al. (2020)
- **VRT response rates** are roughly normally distributed around 0.28–0.35 (28–35% DHI improvement), which aligns with Whitney et al. (2016) meta-analysis findings
- The VR response distribution is wider and shifted slightly higher, consistent with Micarelli et al. (2019) showing superior but more variable results with VR-enhanced rehabilitation

## 5. Correlation Matrix and Heatmap

The correlation matrix reveals the linear relationships between features and targets. Strong inter-feature correlations could indicate redundancy, while feature-target correlations give us an initial signal about predictive power.

In [ ]:
# Correlation heatmap
all_cols = feature_cols + ["vrt_response", "vr_response", "vrt_outcome", "vr_outcome"]
corr = df[all_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1, cbar_kws={"shrink": 0.8})
ax.set_title("Pearson Correlation Matrix — Features & Targets", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Feature-target correlations sorted
print("=== Correlations with VRT Response Rate ===")
vrt_corr = corr["vrt_response"][feature_cols].abs().sort_values(ascending=False)
for feat, val in vrt_corr.items():
    direction = "+" if corr["vrt_response"][feat] > 0 else "-"
    print(f"   {feat:20s}  r = {direction}{val:.3f}")

print("\n=== Correlations with VR Response Rate ===")
vr_corr = corr["vr_response"][feature_cols].abs().sort_values(ascending=False)
for feat, val in vr_corr.items():
    direction = "+" if corr["vr_response"][feat] > 0 else "-"
    print(f"   {feat:20s}  r = {direction}{val:.3f}")

**Correlation Analysis:**

- **Anxiety** is the strongest linear predictor of VRT response rate (negative correlation — higher anxiety → lower response), which aligns with Popkirov et al. (2018) finding that anxiety significantly moderates rehabilitation outcomes
- **Migraine** and **symptom duration** show meaningful negative correlations with both treatment responses
- **Visual sensitivity** shows a notable *positive* correlation with VR response, supporting Micarelli et al. (2019) — VR-based therapy is specifically effective for visually-triggered dizziness
- **Baseline DHI** has weak correlation with response *rates* (as intended), because predicting *how well* someone responds should depend on clinical factors, not just how sick they start
- The correlation between `vrt_outcome` and `baseline_dhi` is still high — this is expected because the absolute outcome score is computed as `baseline * (1 - rate)`, so baseline severity naturally influences the absolute result

## 6. Feature Exploration: New Clinical Variables

A key improvement in this iteration was expanding from 4 features to 8. The additional features — symptom duration, trigger count, migraine, and anxiety disorder — were specifically added to give the model more clinically meaningful predictors and reduce the lopsided dominance of baseline DHI.

### 5.1 Correlation Validation: Simulation vs Published Literature

To verify that the simulated data produces correlations consistent with the literature that informed it, we compare observed Pearson correlations against published values. This is a critical validation step — if the simulation's internal correlations diverge significantly from empirical findings, the downstream ML model would be learning from unrealistic data.

| Relationship | Published r | Source | Simulated r | Match? |
|---|---|---|---|---|
| Anxiety ↔ DHI | ~0.45 | Popkirov et al. (2018) | *computed below* | — |
| Visual sens. ↔ DHI | ~0.38 | Pavlou et al. (2012) | *computed below* | — |
| Duration ↔ VRT response | −0.25 to −0.35 | Bittar & von Söhsten Lins (2015) | *computed below* | — |
| Migraine ↔ VR response | −0.15 to −0.25 | Micarelli et al. (2019) | *computed below* | — |

In [ ]:
# Correlation validation: compare simulated correlations against published values
published = {
    ("anxiety", "baseline_dhi"):      {"r": 0.45, "source": "Popkirov et al. (2018)"},
    ("visual_sens", "baseline_dhi"):  {"r": 0.38, "source": "Pavlou et al. (2012)"},
    ("symptom_duration", "vrt_response"): {"r": -0.30, "source": "Bittar & von Söhsten Lins (2015)"},
    ("migraine", "vr_response"):      {"r": -0.20, "source": "Micarelli et al. (2019)"},
}

print("═══ Correlation Validation: Simulation vs Literature ═══\n")
print(f"{'Relationship':<35s}  {'Published r':>12s}  {'Simulated r':>12s}  {'Δ':>6s}  {'Status'}")
print("─" * 85)

for (col_a, col_b), ref in published.items():
    sim_r = df[col_a].corr(df[col_b])
    delta = abs(sim_r - ref["r"])
    status = "✅ Close" if delta < 0.15 else "⚠️ Divergent"
    label = f"{col_a} ↔ {col_b}"
    print(f"{label:<35s}  {ref['r']:>+12.2f}  {sim_r:>+12.3f}  {delta:>6.3f}  {status}")

print("\nInterpretation: Δ < 0.15 indicates reasonable agreement with published correlations.")
print("Minor deviations are expected — the simulation adds noise and uses a latent-variable")
print("approach rather than directly forcing exact correlation values.")

## Why Response Rate, Not Absolute DHI?

This is the single most important design decision in the project. Training a model on **absolute post-treatment DHI** versus **treatment response rate (% improvement)** produces fundamentally different models with very different clinical utility.

### The Problem with Absolute DHI
If we train the model to predict absolute post-treatment DHI (e.g., "this patient will have DHI = 35 after VRT"), baseline severity dominates at ~94% feature importance. The model essentially learns: `post_DHI ≈ 0.6 × baseline_DHI`. This is mathematically inevitable — patients who start sicker end up with higher absolute scores — but clinically useless. It tells us nothing about *who responds well to treatment*.

### The Solution: Response Rates
By predicting **response rate** (proportion of DHI improvement), we force the model to learn what *drives treatment success*: anxiety, comorbidities, symptom duration, visual sensitivity. Baseline DHI drops to ~4% importance, and clinically meaningful features like anxiety (~49%) and visual sensitivity (~16%) rise to the top.

The comparison below demonstrates this difference.

In [ ]:
# Side-by-side comparison: absolute DHI vs response rate as target

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# LEFT: Train RF on absolute post-treatment DHI (the WRONG way)
rf_abs = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=SEED)
rf_abs.fit(X_train, df.loc[X_train.index, "vrt_outcome"])
abs_imp = pd.Series(rf_abs.feature_importances_, index=feature_cols).sort_values()
abs_imp.plot(kind="barh", ax=axes[0], color="#ef4444")
axes[0].set_title("❌ Predicting Absolute Post-VRT DHI\n(Baseline DHI dominates)", fontsize=12)
axes[0].set_xlabel("Feature Importance (MDI)")
top_feat_abs = abs_imp.idxmax()
axes[0].annotate(f"{top_feat_abs}: {abs_imp.max():.1%} importance",
                 xy=(abs_imp.max(), abs_imp.index.tolist().index(top_feat_abs)),
                 fontsize=10, fontweight="bold", color="#ef4444")

# RIGHT: Train RF on response rate (the RIGHT way)
rf_rate = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=SEED)
rf_rate.fit(X_train, y_train["vrt_response"])
rate_imp = pd.Series(rf_rate.feature_importances_, index=feature_cols).sort_values()
rate_imp.plot(kind="barh", ax=axes[1], color="#10b981")
axes[1].set_title("✅ Predicting Response Rate (% improvement)\n(Clinically meaningful features emerge)", fontsize=12)
axes[1].set_xlabel("Feature Importance (MDI)")

plt.suptitle("Target Variable Choice → Completely Different Models", fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

# Print the dramatic difference
print("\n=== Feature Importance Comparison ===\n")
print(f"{'Feature':<20s}  {'Absolute DHI':>14s}  {'Response Rate':>14s}  {'Change'}")
print("-" * 70)
for feat in feature_cols:
    a = abs_imp.get(feat, 0)
    r = rate_imp.get(feat, 0)
    arrow = "↑↑↑" if r > a + 0.05 else ("↓↓↓" if r < a - 0.05 else "≈")
    print(f"{feat:<20s}  {a:>13.1%}  {r:>13.1%}  {arrow}")
print(f"\n→ Baseline DHI goes from {abs_imp.get('baseline_dhi', 0):.0%} to {rate_imp.get('baseline_dhi', 0):.0%} importance")
print(f"→ Anxiety goes from {abs_imp.get('anxiety', 0):.0%} to {rate_imp.get('anxiety', 0):.0%} importance")
print(f"\nThis is the key insight: the TARGET variable determines what the model learns.")

**Key Takeaway:**

When predicting **absolute post-treatment DHI**, the model learns a trivial relationship: sicker patients have higher scores after treatment too. Baseline DHI dominates at ~94% importance — the model is essentially a pass-through function.

When predicting **response rate** (% improvement), the model learns the *interesting* clinical question: what determines how well a patient responds to treatment? Anxiety emerges as the dominant predictor (~49%), followed by visual sensitivity, migraine comorbidity, and symptom duration — all clinically validated moderators of rehabilitation outcome.

The final DHI score is computed in the app as: `predicted_post_DHI = baseline_DHI × (1 - predicted_rate)`. This preserves clinical utility while ensuring the ML model focuses on the part of the problem that matters.

This single design decision transformed the project from a trivial regression exercise into a clinically interpretable prediction tool.

In [ ]:
# Relationship between new features and treatment response
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Symptom duration vs VRT response (scatter with regression line)
sns.regplot(x="symptom_duration", y="vrt_response", data=df, ax=axes[0, 0],
            scatter_kws={"alpha": 0.3, "s": 15}, color="#3b82f6")
axes[0, 0].set_title("Symptom Duration vs VRT Response")
axes[0, 0].set_xlabel("Symptom Duration (months)")
axes[0, 0].set_ylabel("VRT Response Rate")

# Trigger count vs response (grouped boxplot)
melt = df.melt(id_vars=["trigger_count"], value_vars=["vrt_response", "vr_response"],
               var_name="Treatment", value_name="Response Rate")
melt["Treatment"] = melt["Treatment"].map({"vrt_response": "Traditional VRT", "vr_response": "VR-Based VRT"})
sns.boxplot(x="trigger_count", y="Response Rate", hue="Treatment", data=melt, ax=axes[0, 1],
            palette=["#3b82f6", "#10b981"])
axes[0, 1].set_title("Trigger Count vs Response Rate")
axes[0, 1].legend(fontsize=9)

# Migraine comorbidity effect
melt_mig = df.melt(id_vars=["migraine"], value_vars=["vrt_response", "vr_response"],
                    var_name="Treatment", value_name="Response Rate")
melt_mig["Treatment"] = melt_mig["Treatment"].map({"vrt_response": "Traditional VRT", "vr_response": "VR-Based VRT"})
melt_mig["migraine"] = melt_mig["migraine"].map({0: "No Migraine", 1: "Migraine"})
sns.barplot(x="migraine", y="Response Rate", hue="Treatment", data=melt_mig, ax=axes[1, 0],
            palette=["#3b82f6", "#10b981"], errorbar=("ci", 95))
axes[1, 0].set_title("Migraine Comorbidity Effect on Response")

# Anxiety disorder effect
melt_anx = df.melt(id_vars=["anxiety_disorder"], value_vars=["vrt_response", "vr_response"],
                    var_name="Treatment", value_name="Response Rate")
melt_anx["Treatment"] = melt_anx["Treatment"].map({"vrt_response": "Traditional VRT", "vr_response": "VR-Based VRT"})
melt_anx["anxiety_disorder"] = melt_anx["anxiety_disorder"].map({0: "No Anxiety Disorder", 1: "Anxiety Disorder"})
sns.barplot(x="anxiety_disorder", y="Response Rate", hue="Treatment", data=melt_anx, ax=axes[1, 1],
            palette=["#3b82f6", "#10b981"], errorbar=("ci", 95))
axes[1, 1].set_title("Anxiety Disorder Effect on Response")

plt.suptitle("New Clinical Features vs Treatment Response", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Clinical Feature Analysis:**

- **Symptom duration** shows a clear negative relationship with VRT response — patients with longer symptom histories respond less well, consistent with Bittar & von Söhsten Lins (2015) who found that chronic PPPD (>2 years) has a significantly poorer prognosis
- **Trigger count** shows a dose-response relationship: more triggers = lower treatment response, which makes clinical sense as multi-trigger patients have more complex vestibular dysfunction
- **Migraine comorbidity** substantially reduces VR response rate (the right panel shows a clear difference), consistent with Micarelli et al. (2019) noting that migraine-associated motion sensitivity complicates VR tolerance
- **Anxiety disorder** has a larger effect on traditional VRT than VR-based VRT, supporting the hypothesis that VR may engage anxious patients through its immersive, graded exposure approach

## 7. Exploratory Analysis: Deep-Dive into Treatment Response

In [ ]:
# Which treatment is better for each patient? Compare response rates
df["vr_advantage"] = df["vr_response"] - df["vrt_response"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# VR advantage distribution
sns.histplot(df["vr_advantage"], kde=True, ax=axes[0], color="#8b5cf6", bins=30)
axes[0].axvline(0, color="black", linestyle="--", alpha=0.5)
axes[0].set_title("VR Advantage over Traditional VRT")
axes[0].set_xlabel("VR Response Rate - VRT Response Rate")
pct_vr_wins = (df["vr_advantage"] > 0).mean() * 100
axes[0].annotate(f"VR better: {pct_vr_wins:.0f}%", xy=(0.05, 0.9),
                 xycoords="axes fraction", fontsize=11, fontweight="bold")

# Scatter: anxiety vs response (both treatments)
axes[1].scatter(df["anxiety"], df["vrt_response"], alpha=0.3, s=15, c="#3b82f6", label="VRT")
axes[1].scatter(df["anxiety"], df["vr_response"], alpha=0.3, s=15, c="#10b981", label="VR")
axes[1].set_xlabel("Anxiety Level")
axes[1].set_ylabel("Response Rate")
axes[1].set_title("Anxiety vs Treatment Response")
axes[1].legend()

# Scatter: visual sensitivity vs response
axes[2].scatter(df["visual_sens"], df["vrt_response"], alpha=0.3, s=15, c="#3b82f6", label="VRT")
axes[2].scatter(df["visual_sens"], df["vr_response"], alpha=0.3, s=15, c="#10b981", label="VR")
axes[2].set_xlabel("Visual Sensitivity")
axes[2].set_ylabel("Response Rate")
axes[2].set_title("Visual Sensitivity vs Treatment Response")
axes[2].legend()

plt.tight_layout()
plt.show()

**Treatment Comparison Insights:**

- VR-based VRT shows a higher response rate than traditional VRT for the majority of patients, consistent with meta-analyses showing VR enhancement improves outcomes by 5–10 percentage points
- The VR advantage is most pronounced for patients with high visual sensitivity (rightmost scatter plot), where the VR approach provides targeted desensitisation
- High anxiety diminishes both treatment responses, but the effect is steeper for traditional VRT — suggesting VR may be more robust to anxiety as a confounding factor

## 8. Train/Test Split and Preprocessing

We use an 80/20 train/test split with a fixed random seed for reproducibility. StandardScaler is applied to continuous features for the linear regression model, though tree-based models (Random Forest, Gradient Boosting) don't require scaling.

In [ ]:
# Train/test split
target_cols = ["vrt_response", "vr_response"]
X = df[feature_cols]
y = df[target_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Features:     {X_train.shape[1]}")
print(f"Targets:      {y_train.shape[1]}")

# Standard scaling (for linear regression comparison)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 9. Model Selection Rationale

Three regression models are compared, chosen to represent different complexity levels:

| Model | Why chosen | Strengths | Weaknesses |
|-------|-----------|-----------|------------|
| **Linear Regression** | Baseline model — if a linear model performs well, it suggests the relationships are approximately linear | Highly interpretable, fast, coefficients show direction and magnitude of effects | Cannot capture non-linear interactions between features |
| **Random Forest** | Non-linear ensemble — can capture complex feature interactions without manual engineering | Built-in feature importance, robust to outliers, handles mixed feature types | Less interpretable than linear models, can overfit on small datasets |
| **Gradient Boosting** | Sequential ensemble — often the best performer in tabular data competitions | Strong predictive performance, regularization controls overfitting | Slower to train, more hyperparameters to tune, prone to overfitting |

All models are evaluated using **5-fold cross-validation** on the training set to get robust performance estimates, then final metrics are computed on the held-out test set.

## 10. Model Training and Evaluation

In [ ]:
# Define models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=SEED
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.1, random_state=SEED
    ),
}

# 5-fold cross-validation on training set (VRT response rate as primary target)
print("── 5-Fold Cross-Validation Results (VRT Response Rate) ──\n")
cv_results = {}
for name, model in models.items():
    X_cv = X_train_scaled if name == "Linear Regression" else X_train
    scores = cross_val_score(model, X_cv, y_train["vrt_response"], cv=5, scoring="r2")
    cv_results[name] = {"CV R² Mean": scores.mean(), "CV R² Std": scores.std()}
    print(f"  {name:25s}  R² = {scores.mean():.4f} ± {scores.std():.4f}")

cv_df = pd.DataFrame(cv_results).T
cv_df

In [ ]:
# Train all models on full training set, evaluate on test set
results = []

for name, model in models.items():
    X_tr = X_train_scaled if name == "Linear Regression" else X_train
    X_te = X_test_scaled if name == "Linear Regression" else X_test

    # Train on VRT response
    model.fit(X_tr, y_train["vrt_response"])
    y_pred_vrt = model.predict(X_te)

    results.append({
        "Model": name,
        "Target": "VRT Response",
        "R²": r2_score(y_test["vrt_response"], y_pred_vrt),
        "MAE": mean_absolute_error(y_test["vrt_response"], y_pred_vrt),
        "RMSE": np.sqrt(mean_squared_error(y_test["vrt_response"], y_pred_vrt)),
    })

# Also train multi-output RF (the model used in production)
rf_multi = RandomForestRegressor(
    n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=SEED
)
rf_multi.fit(X_train, y_train)
y_pred_multi = pd.DataFrame(rf_multi.predict(X_test), columns=target_cols, index=y_test.index)

for t in target_cols:
    results.append({
        "Model": "RF Multi-Output",
        "Target": t.replace("_", " ").title(),
        "R²": r2_score(y_test[t], y_pred_multi[t]),
        "MAE": mean_absolute_error(y_test[t], y_pred_multi[t]),
        "RMSE": np.sqrt(mean_squared_error(y_test[t], y_pred_multi[t])),
    })

results_df = pd.DataFrame(results).round(4)
print("── Test-Set Evaluation Metrics ──\n")
results_df

**Model Comparison Discussion:**

- **Linear Regression** achieves the highest R² on single-target VRT response prediction, suggesting the underlying relationships are approximately linear (which makes sense — the simulation uses linear coefficients with noise)
- **Random Forest** performs comparatively well and provides feature importance — this is why it's selected as the production model despite not being the best on R² alone
- **Gradient Boosting** performs similarly but doesn't offer any advantage over RF for this dataset size
- The multi-output RF (used in the Streamlit app) predicts both VRT and VR response rates simultaneously, with slight performance differences between targets

### Why Random Forest over Linear Regression?

Linear Regression wins on cross-validated R² by roughly 0.03–0.04 points. In a standard Kaggle competition, that gap would make LR the obvious choice. But this is a **clinical decision support tool**, where the following considerations matter:

1. **Interpretability through feature importance.** Random Forest provides built-in MDI (Mean Decrease in Impurity) feature importance, which tells clinicians *which factors matter most*. Linear Regression only offers coefficients, which require standardised inputs to compare — less intuitive for clinical audiences.
2. **Robustness to distributional assumptions.** If this system were ever retrained on real clinical data (which would have messy distributions, interaction effects, and non-linearities), a tree-based model would adapt without requiring feature engineering. Linear Regression's slight CV advantage here likely reflects the fact that the simulation itself uses linear coefficients — an advantage that would vanish with real data.
3. **Multi-output prediction.** The RF natively handles predicting both VRT and VR response rates simultaneously with a single model, maintaining consistency between the two predictions. LR would require two separate models.
4. **The cost of 0.04 R².** The R² difference translates to roughly 0.5–1 percentage point of response rate prediction (e.g., predicting 35% vs 35.5% improvement). This is well within the noise margin of clinical measurement and would not change any treatment recommendation.

**Bottom line:** A 0.04 R² sacrifice is a reasonable price to pay for built-in feature importance, multi-output capability, and better generalisation insurance — all of which matter more in a clinical context than marginal predictive accuracy on simulated data.

The R² values in the 0.55–0.65 range indicate the model explains about 55–65% of the variance in treatment response rates. This is a reasonable range for clinical prediction — treatment response is inherently noisy, and our 8 features capture the major determinants but not all individual variation.

In [ ]:
# Residual analysis for all three models (VRT response)
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

model_names = list(models.keys())
for i, (name, model) in enumerate(models.items()):
    X_te = X_test_scaled if name == "Linear Regression" else X_test
    y_pred = model.predict(X_te)
    residuals = y_test["vrt_response"].values - y_pred

    # Residuals vs Predicted
    axes[i, 0].scatter(y_pred, residuals, alpha=0.4, s=20, c="#3b82f6")
    axes[i, 0].axhline(0, color="red", linestyle="--", alpha=0.7)
    axes[i, 0].set_xlabel("Predicted VRT Response Rate")
    axes[i, 0].set_ylabel("Residual")
    axes[i, 0].set_title(f"{name} — Residuals vs Predicted")

    # Residual histogram
    axes[i, 1].hist(residuals, bins=25, color="#3b82f6", edgecolor="white", alpha=0.8)
    axes[i, 1].axvline(0, color="red", linestyle="--", alpha=0.7)
    axes[i, 1].set_xlabel("Residual")
    axes[i, 1].set_ylabel("Count")
    axes[i, 1].set_title(f"{name} — Residual Distribution")
    axes[i, 1].annotate(f"Mean: {residuals.mean():.4f}\nStd: {residuals.std():.4f}",
                         xy=(0.65, 0.8), xycoords="axes fraction", fontsize=10,
                         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.suptitle("Residual Analysis — All Models", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Residual Analysis:**

- All three models show residuals centered near zero (no systematic bias)
- Residual distributions are approximately normal, with no heavy tails
- The scatter plots show no obvious heteroscedasticity — residual variance is roughly constant across the range of predictions
- Random Forest shows slightly more structured residuals (discrete bands), which is typical of tree-based models that partition the feature space into regions

## 11. Feature Importance Analysis

This is perhaps the most important section of the notebook. Feature importance reveals what the model has learned about treatment response prediction, and whether those learned relationships align with clinical knowledge.

In [ ]:
# Feature importance from both tree-based models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest feature importance
rf_model = models["Random Forest"]
rf_imp = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values()
rf_imp.plot(kind="barh", ax=axes[0], color="#3b82f6")
axes[0].set_title("Random Forest — Feature Importance")
axes[0].set_xlabel("Importance (MDI)")

# Gradient Boosting feature importance
gb_model = models["Gradient Boosting"]
gb_imp = pd.Series(gb_model.feature_importances_, index=feature_cols).sort_values()
gb_imp.plot(kind="barh", ax=axes[1], color="#10b981")
axes[1].set_title("Gradient Boosting — Feature Importance")
axes[1].set_xlabel("Importance (MDI)")

plt.suptitle("Feature Importance Comparison — Which Factors Drive Treatment Response?",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print numeric values
print("\nRandom Forest importances:")
for feat, imp in rf_imp.sort_values(ascending=False).items():
    print(f"   {feat:20s}  {imp:.3f}  {'=' * int(imp * 50)}")

print("\nGradient Boosting importances:")
for feat, imp in gb_imp.sort_values(ascending=False).items():
    print(f"   {feat:20s}  {imp:.3f}  {'=' * int(imp * 50)}")

In [ ]:
# Linear Regression coefficients (standardised — show relative effect sizes)
lr_model = models["Linear Regression"]
lr_coefs = pd.Series(lr_model.coef_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#ef4444" if v < 0 else "#10b981" for v in lr_coefs.values]
lr_coefs.plot(kind="barh", ax=ax, color=colors)
ax.set_title("Linear Regression — Standardised Coefficients (VRT Response)")
ax.set_xlabel("Coefficient (effect on response rate)")
ax.axvline(0, color="black", linestyle="-", alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretation: Green = higher value → better treatment response")
print("                Red   = higher value → worse treatment response")

### SHAP Analysis: Individual Prediction Explanations

Global feature importance (MDI) answers: "Which features matter most *overall*?"

But clinicians need patient-level answers: **"Why does the model predict *this* patient will respond poorly to VRT?"**

**SHAP (SHapley Additive exPlanations)** decomposes each individual prediction into per-feature contributions. For a patient predicted to have 22% VRT response, SHAP might show:
- High anxiety: SHAP +8% (pushes prediction toward worse response)
- Long symptom duration: SHAP +5% (pushes toward worse response)
- No migraine: SHAP -3% (pushes toward better response)

This transforms a black-box prediction into an interpretable clinical explanation — exactly what a vestibular physiotherapist would need to discuss treatment options with a patient in a real implementation.

We use `TreeExplainer` (exact SHAP values for tree-based models) to generate:
1. A **waterfall plot** for a sample patient showing which features pushed their prediction up or down
2. A **summary beeswarm plot** showing SHAP values for all test patients

In [ ]:
import shap

# Use the multi-output RF model trained on both targets
explainer = shap.TreeExplainer(rf_multi)
shap_values = explainer(X_test)

# SHAP returns shape (n_samples, n_features, n_outputs) for multi-output
# Extract VRT response SHAP values (first output)
shap_vrt = shap_values[:, :, 0]

# --- Waterfall plot for a sample patient ---
# Pick a patient with moderate-to-poor predicted response (clinically interesting)
pred_vrt = y_pred_multi["vrt_response"].values
sample_idx = np.argmin(np.abs(pred_vrt - np.percentile(pred_vrt, 25)))  # 25th percentile patient

print(f"Sample patient (test index {sample_idx}):")
print(f"  Predicted VRT response rate: {pred_vrt[sample_idx]:.3f} ({pred_vrt[sample_idx]*100:.1f}% DHI reduction)")
print(f"  Actual VRT response rate:    {y_test['vrt_response'].iloc[sample_idx]:.3f}")
print(f"\n  Clinical profile:")
for feat in feature_cols:
    print(f"    {feat:20s} = {X_test.iloc[sample_idx][feat]}")

fig = plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_vrt[sample_idx], max_display=8, show=False)
plt.title("SHAP Waterfall — Why does this patient have a below-average VRT response?", fontsize=12, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# --- SHAP Beeswarm plot: global feature effects across all test patients ---
fig = plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_vrt, max_display=8, show=False)
plt.title("SHAP Beeswarm — Feature Impact on VRT Response Predictions (All Test Patients)", fontsize=12, pad=15)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Each dot = one patient. Colour = feature value (red = high, blue = low).")
print("  Position on x-axis = impact on predicted VRT response rate.")
print("  Anxiety pushes predictions DOWN when high (red dots on left) — high anxiety → worse response.")
print("  Visual sensitivity pushes predictions UP when high — more visual symptoms → better VRT response.")

### Clinical Vignettes: SHAP Interpretation for Three Patient Archetypes

Global SHAP plots show overall patterns, but clinicians think in terms of individual patients. Below, we construct three archetypal PPPD patient profiles and examine **how the model explains each prediction**.

| Profile | Description | Clinical relevance |
|---|---|---|
| **Patient A — "Ideal Candidate"** | Young, low anxiety, short symptom duration, no comorbidities | Expected to respond well to both VRT and VR |
| **Patient B — "Complex Case"** | Older, high anxiety, comorbid migraine and anxiety disorder | Represents the challenging patients seen in tertiary vestibular clinics |
| **Patient C — "VR Advantage"** | High visual sensitivity, moderate anxiety, no migraine | The profile where VR-based VRT is theorised to outperform traditional VRT |

In [ ]:
# Clinical vignettes — construct 3 archetypal patient profiles and interpret with SHAP
vignettes = {
    "Patient A — Ideal Candidate": {
        "age": 30, "baseline_dhi": 45, "anxiety": 2.5, "visual_sens": 3.0,
        "symptom_duration": 6.0, "trigger_count": 2, "migraine": 0, "anxiety_disorder": 0,
    },
    "Patient B — Complex Case": {
        "age": 58, "baseline_dhi": 72, "anxiety": 8.5, "visual_sens": 7.0,
        "symptom_duration": 36.0, "trigger_count": 5, "migraine": 1, "anxiety_disorder": 1,
    },
    "Patient C — VR Advantage": {
        "age": 42, "baseline_dhi": 55, "anxiety": 5.0, "visual_sens": 9.0,
        "symptom_duration": 12.0, "trigger_count": 3, "migraine": 0, "anxiety_disorder": 0,
    },
}

vignette_df = pd.DataFrame(vignettes).T
vignette_X = vignette_df[feature_cols].astype(float)

# Predict with multi-output RF
vignette_pred = pd.DataFrame(
    rf_multi.predict(vignette_X), columns=target_cols, index=vignette_df.index
)

# Compute SHAP values for these patients
vignette_shap = explainer(vignette_X)

# Display predictions
print("═══ Clinical Vignette Predictions ═══\n")
for name in vignettes:
    vrt_r = vignette_pred.loc[name, "vrt_response"]
    vr_r = vignette_pred.loc[name, "vr_response"]
    base = vignettes[name]["baseline_dhi"]
    print(f"  {name}")
    print(f"    VRT response: {vrt_r:.1%} → post-VRT DHI ≈ {base * (1 - vrt_r):.0f}")
    print(f"    VR  response: {vr_r:.1%} → post-VR DHI  ≈ {base * (1 - vr_r):.0f}")
    advantage = "VR" if vr_r > vrt_r else "Traditional VRT"
    print(f"    → Recommended: {advantage} (Δ = {abs(vr_r - vrt_r)*100:.1f} pp)\n")

# SHAP waterfall plots for each vignette (VRT response = output 0)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, name in enumerate(vignettes):
    plt.sca(axes[i])
    shap.plots.waterfall(vignette_shap[i, :, 0], max_display=8, show=False)
    axes[i].set_title(name, fontsize=11)

plt.suptitle("SHAP Waterfall — Per-Patient Treatment Response Explanations (VRT)", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("\nClinical interpretation:")
print("  Patient A: Low anxiety and short duration drive strong predicted response.")
print("  Patient B: High anxiety, migraine, and long duration all push response down.")
print("  Patient C: High visual sensitivity may favour VR over traditional VRT.")

**Feature Importance Discussion:**

By training the model to predict **treatment response rates** (% DHI improvement) rather than absolute post-treatment DHI scores, we've resolved the previous issue where baseline DHI dominated at 94% importance. The current feature importance profile is clinically interpretable:

1. **Anxiety** is the strongest predictor — consistent with extensive literature showing that anxiety is the primary moderator of vestibular rehabilitation outcomes (Popkirov et al., 2018). Patients with high anxiety engage less with VRT exercises and have higher dropout rates.

2. **Visual sensitivity**, **migraine comorbidity**, and **symptom duration** all contribute meaningfully, reflecting that treatment response depends on a constellation of clinical factors, not a single metric.

3. **Baseline DHI** has low importance for predicting the *rate* of response, which is clinically sensible — how sick you are doesn't determine how *well* you respond to treatment. A patient with DHI 80 can respond 40% or 10%, depending on their anxiety, comorbidities, and symptom chronicity.

4. The **linear regression coefficients** show the direction of each effect:
   - Anxiety (negative) → higher anxiety reduces treatment response
   - Migraine (negative) → comorbid migraine reduces response, especially to VR
   - Symptom duration (negative) → chronic symptoms respond less well
   - Visual sensitivity → may be slightly positive for VR response (targeted treatment)

**Design Decision:** We chose to predict response rates because it produces a more informative and balanced model. The absolute post-treatment DHI is then computed as `baseline × (1 - predicted_rate)` in the Streamlit app, preserving clinical utility while ensuring the ML model learns the *interesting* part of the problem.

## 12. Limitations, Reflections, and Future Directions

In [ ]:
# Summary visualisation: actual vs predicted (multi-output RF)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, t in enumerate(target_cols):
    ax = axes[i]
    ax.scatter(y_test[t], y_pred_multi[t], alpha=0.4, s=25, c="#3b82f6")
    lims = [min(y_test[t].min(), y_pred_multi[t].min()) - 0.02,
            max(y_test[t].max(), y_pred_multi[t].max()) + 0.02]
    ax.plot(lims, lims, "r--", alpha=0.7, label="Perfect prediction")
    ax.set_xlabel("Actual Response Rate")
    ax.set_ylabel("Predicted Response Rate")
    title = "Traditional VRT" if "vrt" in t else "VR-Based VRT"
    r2 = r2_score(y_test[t], y_pred_multi[t])
    ax.set_title(f"{title} — Actual vs Predicted (R² = {r2:.3f})")
    ax.legend()

plt.suptitle("Final Model: Multi-Output Random Forest — Prediction Quality", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Honest Assessment of Limitations

**1. The data is simulated — and this matters.**

The core limitation of this project is that all clinical variables (DHI scores, anxiety levels, treatment outcomes) are generated from statistical distributions, not measured from real patients. While the simulation parameters are grounded in published literature, the model is ultimately learning from data we created. This means:
- The discovered relationships reflect our assumptions, not independent empirical patterns
- The model cannot capture unexpected interactions or non-linearities that real data might reveal
- Performance metrics (R², MAE) describe how well the model fits the simulation, not how well it would generalise to real clinical data

**2. The OpenNeuro dataset is from healthy participants.**

The demographics come from 20 healthy subjects in a motion-perception study (Gramann et al., 2021), not PPPD patients. We use these purely for realistic age/sex distributions. A clinical PPPD cohort would differ in age distribution, comorbidity prevalence, and symptom profiles.

**3. Sample size after bootstrap (n=500) is small by ML standards.**

Bootstrap up-sampling creates correlated samples (many rows share the same demographic base), which inflates apparent model performance compared to truly independent observations.

**4. Feature importance reflects the simulation design, not discovered patterns.**

Because we defined the outcome equations, the feature importance is a reflection of the coefficients we chose. In real clinical ML, feature importance would be a genuine discovery. The SHAP analysis in Section 12 makes this even more transparent — individual patient explanations show exactly how each simulated coefficient manifests in predictions.

### What I Learned

1. **Target variable choice matters fundamentally.** Training on absolute post-treatment DHI caused baseline severity to dominate at 94% feature importance — the model was just memorising the input. Switching to response rates produced a clinically meaningful and balanced importance profile.

2. **Transparency builds credibility.** Being explicit about simulation methodology and limitations is more impressive to reviewers than trying to hide them. Every step is documented with literature references.

3. **Clinical domain knowledge drives ML decisions.** The choice of features, outcome formulation, and model interpretation all required understanding of PPPD pathophysiology and vestibular rehabilitation — the ML is only useful in service of the clinical question.

4. **Model selection is context-dependent.** Linear Regression won on R², but Random Forest was the right production choice. In clinical ML, interpretability, robustness, and multi-output capability outweigh marginal accuracy gains on simulated data.

### Future Directions

- **Real clinical data:** Partnership with a vestibular rehabilitation clinic to access anonymised pre/post DHI scores would transform this from a demonstration into a genuine clinical tool
- **Longitudinal modelling:** Treatment response unfolds over weeks — a time-series approach tracking DHI at multiple intervals would capture the trajectory of recovery
- **Additional predictors:** Duration of individual VRT sessions, adherence rates, concurrent medication, and objective vestibular function tests (vHIT, VEMP) could improve prediction
- **Extended SHAP analysis:** The current waterfall and beeswarm plots demonstrate individual explainability. In a production system, SHAP force plots could be embedded directly in the Streamlit dashboard, giving clinicians real-time explanations during patient consultations

### Bootstrap Leakage Analysis

A known risk of bootstrap up-sampling is **data leakage**: because we resample the same 20 original subjects, multiple rows in both train and test sets may share the same demographic base (age, sex). This inflates model performance — the model can partially match test-set rows to near-duplicates it saw during training.

Below, we quantify this effect by comparing the standard random split against a **group-aware split** where all bootstrap copies of the same original subject stay together in either train or test (never both). The performance gap between the two approaches measures the leakage inflation.

In [ ]:
# Bootstrap leakage analysis — quantify performance inflation from shared subjects
from sklearn.model_selection import GroupKFold

# Reload the raw bootstrap index to identify which original subject each row came from
rng_check = np.random.default_rng(42)
boot_idx = rng_check.choice(20, size=500, replace=True)
groups = boot_idx  # group label = original subject index

# Standard 5-fold CV (what we've been using — potential leakage)
rf_check = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=SEED)
standard_cv = cross_val_score(rf_check, X, y["vrt_response"], cv=5, scoring="r2")

# Group-aware 5-fold CV (no leakage — all copies of a subject stay together)
group_cv = cross_val_score(rf_check, X, y["vrt_response"], cv=GroupKFold(n_splits=5),
                            groups=groups, scoring="r2")

print("═══ Bootstrap Leakage Analysis ═══\n")
print(f"  Standard 5-fold CV R²:     {standard_cv.mean():.4f} ± {standard_cv.std():.4f}")
print(f"  Group-aware 5-fold CV R²:  {group_cv.mean():.4f} ± {group_cv.std():.4f}")
print(f"  Performance inflation:     {(standard_cv.mean() - group_cv.mean()):.4f} R² points")
print(f"\n  Interpretation:")

inflation = standard_cv.mean() - group_cv.mean()
if inflation > 0.10:
    print(f"  ⚠️  Substantial leakage detected ({inflation:.1%} R² inflation).")
    print(f"      The standard CV is OVERLY OPTIMISTIC. Group-aware CV gives a more")
    print(f"      honest estimate of how the model would perform on truly unseen subjects.")
elif inflation > 0.03:
    print(f"  ⚠️  Moderate leakage detected ({inflation:.4f} R² inflation).")
    print(f"      Standard CV is somewhat optimistic. Real-world performance would be closer")
    print(f"      to the group-aware estimate.")
else:
    print(f"  ✅  Minimal leakage ({inflation:.4f} R² inflation).")
    print(f"      The clinical simulation layer (DHI, anxiety, outcomes) adds enough noise")
    print(f"      that shared demographics between train/test don't inflate performance much.")

print(f"\n  Note: With only 20 original subjects, GroupKFold has very uneven fold sizes,")
print(f"  which adds variance. The true leakage effect likely lies between the two estimates.")